# 19A2R — Resume Final Cycle-24 AIA Refit

Resume only the final-refit stage after the development loop completed. The interruption was a `KeyboardInterrupt` during the full-pool refit. This notebook preserves the selected best epoch and does not revisit model selection, calibration, thresholding, or Cycle-25.

In [ ]:
from pathlib import Path
import json, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED=20260917
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=True

HOME=Path.home()
PREP=HOME/"aia19_cycle24_cache_prep"
TARGET_MAP=PREP/"cycle24_temporal_targets_local_paths.csv.gz"
OUT=HOME/"aia19_cnn_gru_20260917"
CKPT=OUT/"models"/"cnn_gru_final_refit_resume.pt"
FINAL=OUT/"models"/"cnn_gru_cycle24_final_refit.pt"

EXPECTED_CHANNELS=["aia94","aia131","aia171","aia193","aia211","aia335"]
IMAGE_SIZE=256; BATCH_SIZE=8; NUM_WORKERS=2
LR=3e-4; WEIGHT_DECAY=1e-4; DROPOUT=.30
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:",DEVICE)
print("GPU:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


In [ ]:
hist=pd.read_csv(OUT/"development_history.csv")
best_row=hist.loc[hist["pr_auc"].idxmax()]
best_epoch=int(best_row["epoch"])
norm=json.loads((OUT/"normalisation.json").read_text())
channel_scale=np.asarray(norm["channel_scale"],dtype=np.float32)
channel_mean=np.asarray(norm["channel_mean"],dtype=np.float32)
channel_std=np.asarray(norm["channel_std"],dtype=np.float32)

rows=pd.read_csv(TARGET_MAP)
fit_pool=rows[rows["role"].eq("cycle24_final_refit_pool")].copy()

print("BEST_EPOCH:",best_epoch)
print("BEST_INTERNAL_VAL_PR_AUC:",float(best_row["pr_auc"]))
print("Final refit pool:",len(fit_pool),"positives:",int(fit_pool["label_48h_final"].sum()))


In [ ]:
class TemporalAIADataset(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def _load(self,path):
        with np.load(path,allow_pickle=False) as z:
            x=z["x"]
            ch=[str(v) for v in z["channels"].tolist()]
            if x.shape!=(512,512,6): raise ValueError((path,x.shape))
            if ch!=EXPECTED_CHANNELS: raise ValueError((path,ch))
            if not np.isfinite(x).all(): raise ValueError(f"Nonfinite: {path}")
            x=x.astype(np.float32,copy=False)
        x=np.arcsinh(x/channel_scale.reshape(1,1,6))
        x=(x-channel_mean.reshape(1,1,6))/channel_std.reshape(1,1,6)
        t=torch.from_numpy(x).permute(2,0,1).contiguous()
        return F.interpolate(t.unsqueeze(0),size=(IMAGE_SIZE,IMAGE_SIZE),
                             mode="bilinear",align_corners=False).squeeze(0)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        x=torch.stack([self._load(r["local_tminus288"]),
                       self._load(r["local_tminus192"]),
                       self._load(r["local_tminus96"])])
        y=torch.tensor(float(r["label_48h_final"]),dtype=torch.float32)
        return x,y

class ConvBlock(nn.Module):
    def __init__(self,cin,cout,drop=0):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(cin,cout,3,stride=2,padding=1,bias=False),
            nn.BatchNorm2d(cout),nn.GELU(),
            nn.Conv2d(cout,cout,3,padding=1,bias=False),
            nn.BatchNorm2d(cout),nn.GELU(),
            nn.Dropout2d(drop) if drop else nn.Identity())
    def forward(self,x): return self.net(x)

class FrameCNN(nn.Module):
    def __init__(self,embed=256):
        super().__init__()
        self.enc=nn.Sequential(ConvBlock(6,32,.05),ConvBlock(32,64,.05),
                               ConvBlock(64,128,.10),ConvBlock(128,192,.10),
                               nn.AdaptiveAvgPool2d(1))
        self.proj=nn.Sequential(nn.Flatten(),nn.Linear(192,embed),nn.GELU(),nn.Dropout(DROPOUT))
    def forward(self,x): return self.proj(self.enc(x))

class Model(nn.Module):
    def __init__(self,embed=256,hidden=192):
        super().__init__()
        self.frame=FrameCNN(embed)
        self.gru=nn.GRU(embed,hidden,batch_first=True)
        self.head=nn.Sequential(nn.LayerNorm(hidden),nn.Dropout(DROPOUT),nn.Linear(hidden,1))
    def forward(self,x):
        b,t,c,h,w=x.shape
        z=self.frame(x.reshape(b*t,c,h,w)).reshape(b,t,-1)
        _,hlast=self.gru(z)
        return self.head(hlast[-1]).squeeze(-1)

loader=DataLoader(TemporalAIADataset(fit_pool),batch_size=BATCH_SIZE,shuffle=True,
                  num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)


In [ ]:
model=Model().to(DEVICE)
neg=int((fit_pool["label_48h_final"]==0).sum())
pos=int((fit_pool["label_48h_final"]==1).sum())
criterion=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg/pos],dtype=torch.float32,device=DEVICE))
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
scaler=torch.amp.GradScaler("cuda",enabled=(DEVICE.type=="cuda"))

start_epoch=1
history=[]
if CKPT.exists():
    state=torch.load(CKPT,map_location=DEVICE)
    if int(state["best_epoch_frozen"])!=best_epoch:
        raise RuntimeError("Frozen best_epoch mismatch.")
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scaler.load_state_dict(state["scaler"])
    history=state.get("history",[])
    start_epoch=int(state["epoch"])+1
    print("RESUMING AFTER EPOCH",state["epoch"])
else:
    print("STARTING REFIT FROM EPOCH 1")

for epoch in range(start_epoch,best_epoch+1):
    model.train(); losses=[]; t0=time.time()
    for bi,(x,y) in enumerate(loader,1):
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
            logits=model(x); loss=criterion(logits,y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        losses.append(float(loss.detach().cpu()))
        if bi%500==0:
            print(f"epoch={epoch}/{best_epoch} batch={bi}/{len(loader)} mean_loss={np.mean(losses):.6f}",flush=True)
    rec={"epoch":epoch,"train_loss":float(np.mean(losses)),"seconds":float(time.time()-t0)}
    history.append(rec); print("COMPLETED",rec,flush=True)
    torch.save({"epoch":epoch,"best_epoch_frozen":best_epoch,"model":model.state_dict(),
                "optimizer":optimizer.state_dict(),"scaler":scaler.state_dict(),
                "history":history},CKPT)
    pd.DataFrame(history).to_csv(OUT/"final_refit_history.csv",index=False)

torch.save(model.state_dict(),FINAL)
print("FINAL_MODEL_SAVED:",FINAL)
print("FINAL_REFIT_COMPLETE")


In [ ]:
protocol={
 "status":"TEMPORAL_AIA_CNN_GRU_CYCLE24_BASE_MODEL_FROZEN_PENDING_CALIBRATION",
 "recovery_stage":"19A2R",
 "best_epoch":int(best_epoch),
 "cycle25_used":False,
 "calibration_holdout_used":False,
 "threshold_holdout_used":False,
 "embedded_npz_y_used":False,
 "final_model_path":str(FINAL),
 "scientific_clearance":False
}
(OUT/"protocol_record.json").write_text(json.dumps(protocol,indent=2)+"\n")
print(json.dumps(protocol,indent=2))
